In [ ]:
import os
import gradio as gr
import json
import sqlite3
import datetime
from uuid import uuid4
from openai import OpenAI
from dotenv import load_dotenv

c:\Users\Kenny\Desktop\projects\ML-AI\2026\Q2\gradio_projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
_ = load_dotenv()
openai = OpenAI()
MODEL = "gpt-4.1-mini"

In [19]:
system_message = """You are an airline assistant called Airbot. 
You help customers with their flight bookings, cancellations, 
and general inquiries about flights and services. 
Always be polite and helpful.


In order to successfully book a flight, you must:
1. Ask the customer for their full name, desired destination and travel date.
2. Verify the availability of the desired destination.
3. Confirm the booking details with the customer before finalizing.
4. Finalize the booking and return a unique ticket ID to the customer.


Do not make assumptions on destination availability.
Always ask for necessary details before calling a function.
"""

In [20]:

def get_todays_date():
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

## DB Functions

In [ ]:
def init_db():
    db = sqlite3.connect("airline_data.db")
    cursor = db.cursor()

    # Create bookings table if it doesn't exist
    cursor.execute(
        "CREATE TABLE IF NOT EXISTS bookings (ticket_id TEXT PRIMARY KEY, name TEXT, destination TEXT, date DATE)")

    
    # Create ticket_prices table if it doesn't exist
    cursor.execute("""CREATE TABLE IF NOT EXISTS ticket_prices (
                        location TEXT PRIMARY KEY,
                        price NUMERIC
                    )""")

    # Check if the ticket_prices table is empty
    cursor.execute("SELECT COUNT(*) FROM ticket_prices")
    count = cursor.fetchone()[0]

    if count == 0:
        # Insert initial ticket prices
        prices = [
            ("new york", 300),
            ("los angeles", 250),
            ("chicago", 200),
            ("miami", 350),
            ("dallas", 280),
            ("berlin", 400),
            ("paris", 450),
            ("tokyo", 500),
            ("london", 420),
        ]
        cursor.executemany(
            "INSERT INTO ticket_prices (location, price) VALUES (?, ?)", prices)
        
    db.commit()
    db.close()

In [22]:
init_db()

### Ticket Prices utils

In [ ]:
def run_query(query, params=()):
    db = sqlite3.connect("airline_data.db")
    cursor = db.cursor()
    cursor.execute(query, params)
    results = cursor.fetchall()
    db.commit()
    db.close()
    return results

def set_ticket_price(location, price):
    run_query("INSERT INTO ticket_prices (location, price) VALUES (?, ?) ON CONFLICT(location) DO UPDATE SET price = ?", (location.lower(), price, price))

    return f"Ticket price for {location} has been set to {price}."

def get_ticket_price(location):
    result = run_query("SELECT price FROM ticket_prices WHERE location = ?", (location.lower(),))
    if result:
        return f"The ticket price for {location} is ${result[0][0]}."
    else:
        return f"Price not available for {location}."
    
def check_available_destinations():
    results = run_query("SELECT location FROM ticket_prices")
    destinations = [row[0].title() for row in results]
    return f"Available destinations: {', '.join(destinations)}."

### Booking Utils

In [ ]:
def create_booking(name, destination, date):
    ticket_id = str(uuid4())
    
    # convert date to date object
    date = datetime.datetime.strptime(date, "%Y-%m-%d").date()
    
    run_query("INSERT INTO bookings (ticket_id, name, destination, date) VALUES (?, ?, ?, ?)", (ticket_id, name, destination, date))
    return f"Flight to {destination} on {date} has been booked successfully! Your ticket ID is {ticket_id}."

def cancel_booking(ticket_id):
    run_query("DELETE FROM bookings WHERE ticket_id = ?", (ticket_id,))
    return f"Flight with ticket ID {ticket_id} has been cancelled successfully!"

def check_flight_status(ticket_id):
    result = run_query("SELECT * FROM bookings WHERE ticket_id = ?", (ticket_id,))
    if result:
        return f"Flight with ticket ID {ticket_id} is confirmed and scheduled as planned."
    else:
        return f"No booking found with ticket ID {ticket_id}."

In [ ]:
tools_desc = [
    {
        "name": "get_ticket_price",
        "description": "Get the ticket price for a specific location.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The DESTINATION location to get the ticket price for."
                }
            },
            "required": ["location"]
        }
    },
    {
        "name": "check_available_destinations",
        "description": "Check the available destinations.",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "name": "create_booking",
        "description": "Book a flight for a customer.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {
                    "type": "string",
                    "description": "The name of the customer."
                },
                "destination": {
                    "type": "string",
                    "description": "The destination for the flight."
                },
                "date": {
                    "type": "string",
                    "description": "The date of the flight (YYYY-MM-DD)."
                }
            },
            "required": ["name", "destination", "date"]
        }
    },
    {
        "name": "cancel_booking",
        "description": "Cancel a flight booking using the ticket ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticket_id": {
                    "type": "string",
                    "description": "The ticket ID of the flight booking to cancel."
                }
            },
            "required": ["ticket_id"]
        }
    },
    {
        "name": "check_flight_status",
        "description": "Check the status of a flight booking using the ticket ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticket_id": {
                    "type": "string",
                    "description": "The ticket ID of the flight booking to check."
                }
            },
            "required": ["ticket_id"]
        }
    },
    {
        "name": "get_todays_date",
        "description": "Get today's date.",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    }
]

In [ ]:
tools = [
    {"type": "function", "function": fn_descriptor} for fn_descriptor in tools_desc
]

In [27]:
tool_dict = {
    "get_ticket_price": get_ticket_price,
    "check_available_destinations": check_available_destinations,
    "create_booking": create_booking,
    "cancel_booking": cancel_booking,
    "check_flight_status": check_flight_status,
    "get_todays_date": get_todays_date
}

In [28]:
def handle_tool_calls(message):
    responses = []
    
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        
        if tool_name not in tool_dict.keys():
            return "Unknown tool called"
        
        arguments = json.loads(tool_call.function.arguments)
        
        print(f"Handling tool call: {tool_name} with arguments {arguments}")
        
        response = tool_dict[tool_name](**arguments)
            
        responses.append(
            {
                "role": "tool",
                "content": response,
                "tool_call_id": tool_call.id
            }
        )
        
    return responses

In [33]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]

    messages = [{"role": "system", "content": system_message}] + \
        history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    iterations = 0
    max_iterations = 5  # Prevent infinite loops

    while response.choices[0].finish_reason == "tool_calls" and iterations < max_iterations:
        message = response.choices[0].message
        tool_response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(tool_response)

        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        iterations += 1

    if iterations >= max_iterations:
        return "I'm sorry, but I'm having trouble completing this request. Please try again or contact support."

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Handling tool call: check_available_destinations with arguments {}
Handling tool call: get_todays_date with arguments {}
Handling tool call: check_available_destinations with arguments {}
Handling tool call: get_ticket_price with arguments {'location': 'London'}
Handling tool call: create_booking with arguments {'name': 'Jane Doe', 'destination': 'London', 'date': '2024-04-28'}


C:\Users\Kenny\AppData\Local\Temp\ipykernel_24052\1648399351.py:4: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute(query, params)


Handling tool call: check_available_destinations with arguments {}
Handling tool call: get_ticket_price with arguments {'location': 'Paris'}
Handling tool call: get_ticket_price with arguments {'location': 'Miami'}
Handling tool call: create_booking with arguments {'name': 'Mary Jane', 'destination': 'Paris', 'date': '2024-04-28'}
Handling tool call: create_booking with arguments {'name': 'Anna', 'destination': 'Miami', 'date': '2024-04-28'}


C:\Users\Kenny\AppData\Local\Temp\ipykernel_24052\1648399351.py:4: DeprecationWarning: The default date adapter is deprecated as of Python 3.12; see the sqlite3 documentation for suggested replacement recipes
  cursor.execute(query, params)
